# Supply Chain Demand Forecasting

Predict daily product demand using synthetic sales data.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from prophet import Prophet

df = pd.read_csv("../data/sales_data.csv", parse_dates=["Date"])
df.head()

## EDA: Total Sales per Product

In [ ]:
sales_product = df.groupby('Product')['Sales'].sum()
sales_product.plot(kind='bar', figsize=(10,5), title='Total Sales per Product')
plt.show()

## Feature Engineering

In [ ]:
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['Month'] = df['Date'].dt.month

## Train Random Forest Model

In [ ]:
features = ['Price','Promotion','DayOfWeek','Month']
X = df[features]
y = df['Sales']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
pred = rf.predict(X_test)
print("MAE:", mean_absolute_error(y_test, pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred)))

## Prophet Time Series Forecasting

In [ ]:
# Aggregate daily sales
df_daily = df.groupby('Date')['Sales'].sum().reset_index()
df_daily.columns = ['ds','y']
m = Prophet()
m.fit(df_daily)
future = m.make_future_dataframe(periods=30)
forecast = m.predict(future)
m.plot(forecast)
plt.show()